# 🃏 Previsão de Preço de Cartas Pokémon TCG
## Parte 2 — EDA + Feature Engineering

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

DATA_DIR = Path('data')
with open(DATA_DIR / 'cards_raw.json') as f:
    cards = json.load(f)
print(f'Carregadas {len(cards)} cartas raw')

In [ ]:
# 1. Extração estruturada
def parse_card(c):
    prices = c.get('tcgplayer', {}).get('prices', {})
    holofoil = prices.get('holofoil', {})
    normal = prices.get('normal', {})
    
    # Tentar pegar preço holofoil, se não tiver usa normal como fallback secundário
    target = holofoil.get('market')
    target_type = 'holofoil' if target is not None else None
    if target is None:
        target = normal.get('market')
        target_type = 'normal' if target is not None else None
        
    rel_date = c.get('set', {}).get('releaseDate')
    rel_year = int(rel_date.split('/')[0]) if rel_date and '/' in rel_date else None
    
    subtypes = c.get('subtypes', [])
    types = c.get('types', [])
    pokedex = c.get('nationalPokedexNumbers', [])
    
    hp_str = c.get('hp')
    try:
        hp = float(hp_str) if hp_str else None
    except:
        hp = None
        
    return {
        'id': c['id'],
        'name': c.get('name'),
        'hp': hp,
        'supertype': c.get('supertype', 'Unknown'),
        'subtypes_str': ', '.join(subtypes),
        'subtypes_count': len(subtypes),
        'types_str': ', '.join(types),
        'primary_type': types[0] if types else 'Colorless',
        'rarity': c.get('rarity', 'Unknown'),
        'artist': c.get('artist', 'Unknown'),
        'set_id': c.get('set', {}).get('id', 'Unknown'),
        'set_name': c.get('set', {}).get('name', 'Unknown'),
        'set_series': c.get('set', {}).get('series', 'Unknown'),
        'set_printed_total': c.get('set', {}).get('printedTotal'),
        'release_year': rel_year,
        'card_age_years': (2026 - rel_year) if rel_year else None,
        'pokedex_number': pokedex[0] if pokedex else None,
        'is_gen1': (pokedex[0] <= 151) if pokedex else False,
        'target_price': target,
        'price_type': target_type
    }

df = pd.DataFrame([parse_card(c) for c in cards])
df = df[df['target_price'].notna() & (df['target_price'] > 0)].copy()
df['log_target_price'] = np.log1p(df['target_price'])
print(f'Cartas válidas com preço: {len(df)}')

In [ ]:
# 2. Estatísticas do Target
print('=== ESTATÍSTICAS DE PREÇO ($USD) ===')
print(df['target_price'].describe())
print('\nCartas por tipo de preço:')
print(df['price_type'].value_counts())

In [ ]:
# 3. Relação Raridade x Preço Média
rarity_stats = df.groupby('rarity')['target_price'].agg(['count', 'mean', 'median', 'max']).sort_values('median', ascending=False)
print('=== PREÇO MÉDIO/MEDIANA POR RARIDADE ===')
print(rarity_stats)

In [ ]:
# 4. Salvar dataset processado
df.to_csv(DATA_DIR / 'cards_processed.csv', index=False)
print(f'✅ Dataset processado salvo em {DATA_DIR / "cards_processed.csv"} ({df.shape[0]} linhas, {df.shape[1]} colunas)')